# Multivariate Linear Regression — AQ-10 Autism Screening Result

**Mission:** predict a person's AQ-10 autism-screening `result` using only
accessible, non-behavioral information (age, family history of autism, country
of residence, jaundice at birth, etc.) — without needing the full 10-question
behavioral interview (A1–A10). This supports lightweight pre-screening in
regions or families that don't have easy access to the full assessment.

Built from three merged datasets (Fadi Thabtah, UCI ML Repository): Adult,
Child, and Adolescent autism screening data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib

SEQ_BLUE = "#2a78d6"

## Step 1 — Load and merge

In [2]:
adult = pd.read_csv('Autism_Adult_Data.csv')
adult['age_group'] = 'adult'

child = pd.read_csv('Autism_Child_Data.csv')
child['age_group'] = 'child'

adolescent = pd.read_csv('Autism_Adolescent_Data.csv')
adolescent['age_group'] = 'adolescent'

df = pd.concat([adult, child, adolescent], ignore_index=True)

print(df.shape)
df.info()

(1100, 23)
<class 'pandas.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               1100 non-null   int64 
 1   A1_Score         1100 non-null   int64 
 2   A2_Score         1100 non-null   int64 
 3   A3_Score         1100 non-null   int64 
 4   A4_Score         1100 non-null   int64 
 5   A5_Score         1100 non-null   int64 
 6   A6_Score         1100 non-null   int64 
 7   A7_Score         1100 non-null   int64 
 8   A8_Score         1100 non-null   int64 
 9   A9_Score         1100 non-null   int64 
 10  A10_Score        1100 non-null   int64 
 11  age              1100 non-null   object
 12  gender           1100 non-null   str   
 13  ethnicity        1100 non-null   str   
 14  jundice          1100 non-null   str   
 15  austim           1100 non-null   str   
 16  contry_of_res    1100 non-null   str   
 17  used_app_before  1100 non-null   

## Step 2 — Clean

In [3]:
df = df.drop(columns=['id', 'Class/ASD'])

df['ethnicity'] = df['ethnicity'].replace('?', 'Unknown')
df['relation'] = df['relation'].replace('?', 'Unknown')

# age arrives as a mix of str/int across the 3 source files and has a
# handful of '?' placeholders (adult + child); coerce to numeric and
# drop the few rows that can't be parsed.
df['age'] = pd.to_numeric(df['age'], errors='coerce')
print('rows dropped for unparseable age:', df['age'].isna().sum())
df = df.dropna(subset=['age']).reset_index(drop=True)

df.shape

rows dropped for unparseable age: 6


(1094, 21)

In [4]:
a_score_cols = [f'A{i}_Score' for i in range(1, 11)]
df = df.drop(columns=a_score_cols)
df.shape

(1094, 11)

### Why A1–A10 are dropped

`result` (the AQ-10 score) is the arithmetic sum of the ten `A1_Score`–
`A10_Score` answers. Keeping those columns as features would let the model
just re-add them — a deterministic, trivial mapping with near-zero error that
says nothing about screening someone *without* the behavioral checklist.
Since the mission is to predict `result` from accessible, non-behavioral
information, the ten item scores are dropped entirely, along with
`Class/ASD` (a direct threshold of `result`, i.e. leakage) and `id` (a row
identifier with no predictive value).